# File: 5_torch_data_ops.ipynb

Bu notebook PyTorch'da veri işlemleri, Dataloader ve MNIST veri setiyle çok sınıflı sınıflandırma örneklerini içermektedir.
----------------------------------------------------------------

In [ ]:
# PyTorch ana kütüphanesini içe aktarır - tensor işlemleri ve derin öğrenme için
import torch

# PyTorch'un sinir ağı modülünü içe aktarır - katmanlar, aktivasyon fonksiyonları, kayıp fonksiyonları
import torch.nn as nn

# Stokastik Gradyan İnişi optimizasyonu için
from torch.optim import SGD

# PyTorch veri yükleme yardımcıları - Dataset ve DataLoader sınıfları
from torch.utils.data import Dataset, DataLoader

# Sayısal hesaplamalar ve dizi işlemleri için NumPy kütüphanesi
import numpy as np

# Grafik ve görselleştirme için matplotlib kütüphanesi
import matplotlib.pyplot as plt

# Eğitim/test verisi bölme fonksiyonu
from sklearn.model_selection import train_test_split

# Model performans değerlendirme metrikleri
from sklearn.metrics import accuracy_score

# MNIST veri setini okumak için özel kütüphane
import idx2numpy

## PyTorch Veri İşleme Mimarisi ve Dataset/DataLoader Ekosistemi

### PyTorch'un Veri Yükleme Felsefesi

PyTorch'un veri yükleme yaklaşımı, **composability** (birleştirilebilirlik) ve **efficiency** (verimlilik) prensipleri üzerine kuruludur. Bu sistem, büyük ölçekli makine öğrenmesi projelerinde karşılaşılan temel zorlukları çözmek için tasarlanmıştır:

#### 1. **Bellek Yönetimi Problemleri**
```
Geleneksel Yaklaşım (Naif):
├── Tüm veriyi RAM'e yükle
├── Bellek tükenmesi (OOM - Out of Memory)
└── 64GB+ RAM gereksinimi

PyTorch Dataset Yaklaşımı:
├── Lazy loading (ihtiyaç halinde yükleme)
├── Streaming data access
├── Memory-mapped files desteği
└── 8GB RAM ile terabayt veriler işlenebilir
```

#### 2. **I/O Bottleneck Çözümü**
```
Problem: Disk okuma hızı << GPU hesaplama hızı
├── GPU beklemede (underutilization)
├── Training time'ın %70'i I/O wait
└── Pahalı GPU kaynaklarının verimsiz kullanımı

DataLoader Çözümü:
├── Multi-process data loading
├── Prefetching (önceden yükleme)
├── Background data preparation
└── GPU utilization %95+
```

#### 3. **Veri Transformasyon Pipeline'ı**
```
Raw Data → Augmentation → Normalization → Tensor → GPU
    ↑           ↑            ↑           ↑       ↑
    Disk     CPU-intensive  Vectorized  Memory  CUDA
             ops            ops         Transfer
```

In [ ]:
# MNIST veri seti dosya yolu
MNIST_DIR = "mnist/"

# MNIST eğitim görüntülerini oku (60000 adet 28x28 piksel görüntü)
# idx2numpy.convert_from_file(): IDX formatındaki dosyayı NumPy dizisine dönüştürür
X_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-images-idx3-ubyte")

# Görüntüleri düzleştir ve normalize et: (60000, 28, 28) -> (60000, 784) ve [0,255] -> [0,1]
# reshape(60000, -1): -1 otomatik boyut hesaplama (28*28=784)
# /255.0: Piksel değerlerini 0-1 aralığına normalize et
X_mnist = X_mnist.reshape(60000, -1) / 255.0

# MNIST eğitim etiketlerini oku (0-9 arası rakam etiketleri)
y_mnist = idx2numpy.convert_from_file(MNIST_DIR + "train-labels-idx1-ubyte")

# İlk 5 etiketin değerlerini kontrol et
print(y_mnist[:5])

# Veri setindeki benzersiz etiketleri kontrol et (0-9 arası olmalı)
print(np.unique(y_mnist))

The number 42 is, in The Hitchhiker's Guide to the Galaxy by Douglas Adams, the "Answer to the Ultimate Question of Life, the Universe, and Everything", calculated by an enormous supercomputer named Deep Thought over a period of 7.5 million years. Unfortunately, no one knows what the question is.

In [ ]:
# MNIST veri setini eğitim ve test setlerine böl
# - test_size=0.2: Verinin %20'si test için ayrılır (%80 eğitim)
# - random_state=42: Aynı bölünmeyi her seferinde elde etmek için seed
X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42)  

# NumPy dizilerini PyTorch tensörlerine dönüştür
# from_numpy(): NumPy dizisini PyTorch tensörüne dönüştürür (hafıza paylaşımlı)
# astype(np.float32): Veri tipini 32-bit float'a dönüştür (GPU uyumluluğu için)
x = torch.from_numpy(X_train.astype(np.float32))

# Etiketleri int64 (long) tipine dönüştür (CrossEntropyLoss için gerekli)
y = torch.from_numpy(y_train.astype(np.int64))

# Tensör boyutlarını kontrol et
print(x.shape)  # (48000, 784) - 48000 eğitim örneği, 784 özellik
print(y.shape)  # (48000,) - 48000 etiket

Torch Dataset'ten türeyen kendi classımızı oluşturma

In [ ]:
# PyTorch Dataset sınıfından türetilen özel MNIST veri seti sınıfı
class MnistDataset(Dataset):
    def __init__(self, X, y):
        # Veri seti başlangıç fonksiyonu - veri yükleme ve ön işleme
        
        # NumPy dizilerini PyTorch tensörlerine dönüştür
        # astype(np.float32): Özellikler için 32-bit float veri tipi
        self.x = torch.from_numpy(X.astype(np.float32))
        
        # astype(np.int64): Etiketler için 64-bit integer veri tipi (sınıflandırma için)
        self.y = torch.from_numpy(y.astype(np.int64))

    def __getitem__(self, index):
        # Belirli bir indeksteki veri örneğini döndür
        # Bu metod, dataset[index] sözdiziminin çalışmasını sağlar
        # Dönen değer: (özellik_vektörü, etiket) çifti
        return self.x[index], self.y[index]

    def __len__(self):
        # Veri setindeki toplam örnek sayısını döndür
        # Bu metod, len(dataset) sözdiziminin çalışmasını sağlar
        # shape[0]: İlk boyut (örnek sayısı)
        return self.x.shape[0]

In [ ]:
# MnistDataset sınıfından bir örnek oluştur
dataset = MnistDataset(X_train, y_train)

# 42 numaralı indeksteki veri örneğini al
# Douglas Adams'ın "Hitchhiker's Guide to the Galaxy"sinde yaşam, evren ve her şeyin anlamının cevabı
first_x, first_y = dataset[42]

# Seçilen örneği görselleştir
plt.figure(figsize=(3, 3))

# Etiket değerini başlık olarak göster
# .item(): Tensör skalarını Python sayısına dönüştürür
plt.title(first_y.item())

# 784 boyutlu vektörü 28x28 görüntüye dönüştür ve göster
# reshape(28, 28): Düzleştirilmiş vektörü orijinal 2D forma döndür
# cmap="gray": Gri tonlama renk haritası
plt.imshow(first_x.reshape(28, 28), cmap="gray")

# Grafiği göster
plt.show()

Kendi MnistDataset sınıfımızı kullanan DataLoader nesnesi oluşturma

### Model Eğitim Döngüsü: İleri Seviye Optimizasyon Teknikleri

#### CrossEntropyLoss Derinlemesine Analizi

**Matematiksel Formülasyon ve Numerical Stability:**
```python
class StableCrossEntropyLoss:
    """
    Numerically stable CrossEntropyLoss implementation
    """
    
    @staticmethod
    def manual_cross_entropy(logits, targets):
        """
        Manual implementation showing internal workings
        """
        # Step 1: LogSumExp for numerical stability
        max_logits = torch.max(logits, dim=1, keepdim=True)[0]
        shifted_logits = logits - max_logits
        
        # Step 2: Compute log probabilities
        log_sum_exp = torch.log(torch.sum(torch.exp(shifted_logits), dim=1, keepdim=True))
        log_probs = shifted_logits - log_sum_exp
        
        # Step 3: Negative log likelihood
        batch_size = logits.size(0)
        loss = -log_probs[range(batch_size), targets]
        
        return loss.mean()
    
    @staticmethod
    def compare_implementations():
        """
        Compare numerical stability
        """
        # Extreme logits (potential overflow)
        logits = torch.tensor([[1000., 1001., 999.]], dtype=torch.float32)
        targets = torch.tensor([1])
        
        # PyTorch implementation (stable)
        pytorch_loss = F.cross_entropy(logits, targets)
        
        # Naive implementation (unstable)
        naive_softmax = F.softmax(logits, dim=1)
        naive_loss = -torch.log(naive_softmax[0, targets[0]])
        
        # Manual stable implementation
        stable_loss = StableCrossEntropyLoss.manual_cross_entropy(logits, targets)
        
        return {
            'pytorch_loss': pytorch_loss.item(),
            'naive_loss': naive_loss.item(),  # May be inf or nan
            'stable_loss': stable_loss.item()
        }
```

**Label Smoothing ve Regularization:**
```python
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Label smoothing for better generalization
    """
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, logits, targets):
        """
        Apply label smoothing:
        y_smooth = (1-ε)y_true + ε/K
        where ε is smoothing factor, K is number of classes
        """
        log_probs = F.log_softmax(logits, dim=1)
        
        # One-hot encoding with smoothing
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
        
        # KL divergence between smooth target and predicted distribution
        return torch.mean(torch.sum(-true_dist * log_probs, dim=1))
```

#### Advanced SGD: Momentum ve Adaptive Learning

**SGD with Momentum Detaylı Analizi:**
```python
class MomentumSGD:
    """
    Manual implementation of SGD with momentum
    """
    def __init__(self, parameters, lr=0.01, momentum=0.9, weight_decay=0):
        self.parameters = list(parameters)
        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        
        # Initialize momentum buffers
        self.velocity = [torch.zeros_like(p) for p in self.parameters]
    
    def step(self):
        for i, param in enumerate(self.parameters):
            if param.grad is None:
                continue
            
            grad = param.grad.data
            
            # L2 weight decay
            if self.weight_decay != 0:
                grad = grad.add(param.data, alpha=self.weight_decay)
            
            # Momentum update
            self.velocity[i] = self.momentum * self.velocity[i] + grad
            
            # Parameter update
            param.data = param.data - self.lr * self.velocity[i]
    
    def zero_grad(self):
        for param in self.parameters:
            if param.grad is not None:
                param.grad.zero_()

# Momentum effect analysis:
"""
Without momentum:
θₜ₊₁ = θₜ - η∇L(θₜ)
- Can oscillate in narrow valleys
- Slow convergence on plateaus

With momentum:
vₜ₊₁ = βvₜ + ∇L(θₜ)
θₜ₊₁ = θₜ - ηvₜ₊₁
- Smooths out oscillations
- Accelerates in consistent directions
- β=0.9 means 90% of previous velocity retained
"""
```

#### Batch Size Effects: Mini-batch vs Full Batch

**Gradient Noise Analysis:**
```python
class GradientNoiseAnalyzer:
    """
    Analyze the effect of batch size on gradient noise
    """
    
    @staticmethod
    def compute_gradient_variance(model, dataloader, num_samples=100):
        """
        Estimate gradient variance for different batch sizes
        """
        gradients = []
        
        for i, (X, y) in enumerate(dataloader):
            if i >= num_samples:
                break
                
            # Compute gradient for this batch
            output = model(X)
            loss = F.cross_entropy(output, y)
            
            grad_dict = torch.autograd.grad(loss, model.parameters(), retain_graph=False)
            gradients.append([g.clone() for g in grad_dict])
        
        # Compute variance across batches
        variances = []
        for param_idx in range(len(gradients[0])):
            param_grads = torch.stack([g[param_idx] for g in gradients])
            variance = torch.var(param_grads, dim=0).mean().item()
            variances.append(variance)
        
        return variances
    
    @staticmethod
    def generalization_analysis():
        """
        Theoretical analysis of batch size effects
        """
        return {
            'small_batch_benefits': [
                'Higher gradient noise → better exploration',
                'Implicit regularization effect',
                'Better generalization (flatter minima)',
                'More frequent parameter updates'
            ],
            'large_batch_benefits': [
                'More accurate gradient estimates',
                'Better convergence guarantees',
                'More efficient GPU utilization',
                'Stable training dynamics'
            ],
            'optimal_strategy': 'Progressive batch size increase during training'
        }
```

#### Memory Optimization Techniques

**Gradient Accumulation:**
```python
class GradientAccumulator:
    """
    Simulate large batch sizes with gradient accumulation
    """
    def __init__(self, model, optimizer, accumulation_steps=4):
        self.model = model
        self.optimizer = optimizer
        self.accumulation_steps = accumulation_steps
        self.current_step = 0
    
    def accumulate_step(self, loss):
        """
        Accumulate gradients without updating parameters
        """
        # Scale loss by accumulation steps
        scaled_loss = loss / self.accumulation_steps
        scaled_loss.backward()
        
        self.current_step += 1
        
        # Update parameters when accumulation is complete
        if self.current_step % self.accumulation_steps == 0:
            self.optimizer.step()
            self.optimizer.zero_grad()
    
    def finalize_epoch(self):
        """
        Handle remaining gradients at epoch end
        """
        if self.current_step % self.accumulation_steps != 0:
            self.optimizer.step()
            self.optimizer.zero_grad()

# Memory analysis:
"""
Regular training (batch_size=256):
Memory = 256 * (forward_activations + backward_gradients)

Gradient accumulation (effective_batch_size=256, micro_batch=64):
Memory = 64 * (forward_activations + backward_gradients)
Effective batch size = 256 (same convergence)
Memory reduction = 4x
"""
```

#### Learning Rate Scheduling

**Cosine Annealing Implementation:**
```python
class CosineAnnealingLR:
    """
    Cosine annealing learning rate scheduler
    """
    def __init__(self, optimizer, T_max, eta_min=0):
        self.optimizer = optimizer
        self.T_max = T_max
        self.eta_min = eta_min
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]
        self.current_epoch = 0
    
    def step(self):
        """
        Update learning rate using cosine annealing
        """
        for param_group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            # Cosine annealing formula
            param_group['lr'] = self.eta_min + (base_lr - self.eta_min) * \
                               (1 + math.cos(math.pi * self.current_epoch / self.T_max)) / 2
        
        self.current_epoch += 1
    
    def get_lr_analysis(self):
        """
        Analyze learning rate schedule
        """
        epochs = list(range(self.T_max))
        lrs = []
        
        for epoch in epochs:
            lr = self.eta_min + (self.base_lrs[0] - self.eta_min) * \
                 (1 + math.cos(math.pi * epoch / self.T_max)) / 2
            lrs.append(lr)
        
        return epochs, lrs

# Warm restart strategy:
"""
SGDR (Stochastic Gradient Descent with Warm Restarts):
- Periodically restart learning rate to high value
- Allows model to escape local minima
- T_max increases with each restart: T₀, 2*T₀, 4*T₀, ...
"""
```

In [ ]:
# Dataset'i kullanarak DataLoader oluştur
# DataLoader: Mini-batch eğitimi, veri karıştırma ve paralel yükleme sağlar
data_loader = DataLoader(
    dataset=dataset,     # Kullanılacak veri seti
    batch_size=4,        # Her batch'te 4 örnek olsun
    shuffle=True         # Her epoch'ta veriyi karıştır (overfitting'i önler)
)

# DataLoader'dan iterator oluştur
# iter(): DataLoader'ı yinelenebilir nesneye dönüştürür
data_iterator = iter(data_loader)

# İlk batch'i al
# next(): Iterator'dan bir sonraki öğeyi (batch'i) getirir
x, y = next(data_iterator)

# Batch boyutlarını kontrol et
print(f"x.shape: {x.shape}, y.shape:{y.shape}")
# x.shape: (4, 784) - 4 örnek, her biri 784 özellikli
# y.shape: (4,) - 4 etiket

In [ ]:
# Batch'teki 4. örneği görselleştir (indeks 3)
plt.figure(figsize=(3, 3))

# y[3]: Batch'teki 4. örneğin etiketini al
# .item(): Tensör skalarını Python sayısına dönüştür
plt.title(y[3].item())

# x[3]: Batch'teki 4. örneğin özellik vektörünü al
# reshape(28, 28): 784 boyutlu vektörü 28x28 görüntüye dönüştür
plt.imshow(x[3].reshape(28, 28), cmap="gray")

# Grafiği göster
plt.show()

Multi-class classification modeli oluşturma 

Model Mimarisi:
1. Giriş katmanı: 784 -> 32
2. Sonuçları sigmoid ile dönüştürme
3. Çıkış katmanı: 32 -> 10

In [ ]:
# Çok sınıflı MNIST sınıflandırması için sinir ağı modeli
class MNistClassifier(nn.Module):
    def __init__(self):
        # Üst sınıfın (nn.Module) __init__ metodunu çağır
        super().__init__()
        
        # Girdi katmanı: 784 nörondan 32 nörona lineer dönüşüm
        # 784: Düzleştirilmiş 28x28 piksel görüntü
        # 32: Gizli katman boyutu (hiperparametre)
        self.input_layer = nn.Linear(784, 32)
        
        # Sigmoid aktivasyon fonksiyonu
        # σ(x) = 1/(1+e^(-x)) - çıkışı 0-1 aralığında sıkıştırır
        self.activation = nn.Sigmoid()
        
        # Çıkış katmanı: 32 nörondan 10 sınıfa lineer dönüşüm
        # 10: MNIST'te 10 farklı rakam sınıfı (0-9)
        self.output_layer = nn.Linear(32, 10)
        
    def forward(self, x):
        # İleri yayılım (forward pass) fonksiyonu
        
        # Girdi katmanından geçir: 784 -> 32
        x = self.input_layer(x)
        
        # Sigmoid aktivasyonu uygula (non-linearity)
        x = self.activation(x)
        
        # Çıkış katmanından geçir: 32 -> 10
        # Bu katmandan çıkan değerler "logit"ler (ham skorlar)
        x = self.output_layer(x)
        
        # Logit değerlerini döndür (CrossEntropyLoss kendi softmax'ını uygular)
        return x

Model Eğitimi

In [ ]:
# Model eğitimi için gerekli bileşenleri hazırla

# DataLoader oluştur - daha büyük batch boyutu ile
data_loader = DataLoader(
    dataset=dataset,     # MNIST veri seti
    batch_size=256,      # Her batch'te 256 örnek (daha verimli GPU kullanımı)
    shuffle=True         # Veriyi her epoch'ta karıştır
)

# Model örneği oluştur
model = MNistClassifier()

# Çok sınıflı sınıflandırma için kayıp fonksiyonu
# CrossEntropyLoss = Softmax + NegativeLogLikelihood
# Logit değerlerini doğrudan kabul eder (softmax uygulamaya gerek yok)
criterion = nn.CrossEntropyLoss()

# Stokastik Gradyan İnişi optimizeri
# lr=0.01: Öğrenme hızı (learning rate)
optimizer = SGD(model.parameters(), lr=0.01)

# Eğitim sürecini takip etmek için kayıp değerlerini sakla
losses = []

# 40 epoch boyunca model eğitimi
for epoch in range(40):
    # Her epoch'ta tüm batch'ler üzerinde döngü
    for i, (X, y) in enumerate(data_loader):
        # İleri yayılım: Model tahminleri üret
        y_hat = model(X)
        
        # Kayıp hesapla: Tahmin vs gerçek etiketler
        loss = criterion(y_hat, y)
        
        # Geri yayılım: Gradyanları hesapla
        loss.backward()
        
        # Parametreleri güncelle
        optimizer.step()
        
        # Gradyanları sıfırla (bir sonraki iterasyon için)
        optimizer.zero_grad()

        # Kayıp değerini kaydet
        # .item(): Tensör skalarını Python sayısına dönüştür
        losses.append(loss.item())

        # Her 10 batch'te bir ilerleme raporu yazdır
        if (i+1) % 10 == 0:
            print(f"epoch: {epoch + 1}, batch: {i + 1}, train loss: {loss.item()}")

In [ ]:
# Eğitilmiş modelin parametrelerini incele
for name, param in model.named_parameters():
    # requires_grad=True olan parametreleri (öğrenilebilir parametreler) kontrol et
    if param.requires_grad:
        # Parametre adı ve boyutunu yazdır
        print(name, param.shape)
        # Örnek çıktı:
        # input_layer.weight torch.Size([32, 784]) - Girdi katmanı ağırlık matrisi
        # input_layer.bias torch.Size([32]) - Girdi katmanı bias vektörü
        # output_layer.weight torch.Size([10, 32]) - Çıkış katmanı ağırlık matrisi
        # output_layer.bias torch.Size([10]) - Çıkış katmanı bias vektörü

Hata Görselleştirme

In [ ]:
# Eğitim sürecinin kayıp değerlerini görselleştir
plt.figure(figsize=(12, 4))

# Grafik başlığı
plt.title("Loss")

# Kayıp değerlerini zaman serisine karşı çiz
# losses: Her batch için kayıp değerleri listesi
plt.plot(losses)

# X ekseni etiketi (batch sayısı/zaman)
plt.xlabel("Time")

# Y ekseni etiketi (Cross Entropy Loss değeri)
plt.ylabel("BCE Loss")

# Grafiği göster
plt.show()

In [ ]:
# Eğitimin başlangıcındaki kayıp değerlerini yakından incele
first_n = 200  # İlk 200 kayıp değeri

plt.figure(figsize=(12, 6))

# Grafik başlığı - dinamik olarak first_n değerini göster
plt.title(f"First {first_n} Loss Values")

# losses listesinin ilk 200 değerini çiz
# [:first_n]: Liste dilimlemesi - indeks 0'dan first_n'ye kadar
plt.plot(losses[:first_n])

# Eksen etiketleri
plt.xlabel("Time")
plt.ylabel("BCE Loss")

# Grafiği göster
plt.show()

Test Seti ile Modelin Başarısını Hesaplama

In [ ]:
# Test veri seti üzerinde model performansını değerlendir

# Gradyan hesaplamasını devre dışı bırak (değerlendirme modu)
# Bu, hafıza kullanımını azaltır ve hesaplamaları hızlandırır
with torch.no_grad():
    # Test verilerini PyTorch tensörüne dönüştür
    # from_numpy(): NumPy dizisini PyTorch tensörüne dönüştür
    # astype(np.float32): Veri tipini 32-bit float'a çevir
    test_tensor = torch.from_numpy(X_test.astype(np.float32))
    
    # Model ile test verisi üzerinde tahmin yap
    # y_predictions: Her örnek için 10 sınıfa ait logit değerleri
    y_predictions = model(test_tensor)

# İlk 3 örneğin logit değerlerini göster
# Her satır bir örnek, her sütun bir sınıf (0-9) skorunu temsil eder
print(y_predictions[:3])

Logitlerden ağırlıklarından sınıflara...

In [ ]:
# Logit değerlerini sınıf tahminlerine dönüştür

# torch.argmax(): En yüksek değere sahip indeksi bulur
# dim=1: Her satır (örnek) için sütunlar (sınıflar) arasında maksimum bulur
# Sonuç: Her örnek için en olası sınıfın indeksi (0-9 arası)
y_predicted_classes = torch.argmax(y_predictions, dim=1)

# Örnek:
# y_predictions[0] = [0.1, 0.05, 0.8, 0.02, ...] -> argmax = 2 (indeks 2'de maksimum)
# Bu, modelin ilk örneği "2" rakamı olarak sınıflandırdığı anlamına gelir

In [ ]:
# İlk 3 sınıf tahminini kontrol et
# Bu değerler 0-9 arası tamsayılar olmalı (MNIST sınıfları)
y_predicted_classes[:3]

In [ ]:
# Gerçek test etiketlerinin ilk 3 değerini kontrol et
# Model tahminleriyle karşılaştırmak için
y_test[:3]

In [ ]:
# Model performansını accuracy metrifi ile hesapla

# accuracy_score(): Doğru tahmin edilen örneklerin toplam örnek sayısına oranı
# y_test: Gerçek etiketler (NumPy dizisi)
# y_predicted_classes: Model tahminleri (PyTorch tensörü - otomatik NumPy'a dönüştürülür)
# Formül: Accuracy = (Doğru Tahminler) / (Toplam Tahminler)
acc = accuracy_score(y_test, y_predicted_classes)

# Accuracy değerini yazdır
# 0.0-1.0 arası değer, 1.0 = %100 doğruluk
print(acc)